# 01 · Descripción y calidad del dataset GH-AW

Dataset propio: **[luchosqi/gh-aw-workflows](https://huggingface.co/datasets/luchosqi/gh-aw-workflows)**,
construido con [Miner](https://github.com/Luchosqi/Miner) (`miner dataset`) a partir de repositorios de
GitHub que usan **GitHub Agentic Workflows (GH-AW)**.

Este notebook cubre:

1. Origen y carga de los datos
2. Descripción de las tablas y sus relaciones
3. Revisión de calidad
4. Tratamiento de los problemas encontrados

Las tablas resultantes de este notebook se guardan en `eda/data/processed/` y son las que
consume `02_exploracion_y_hallazgos.ipynb`.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATASET = "luchosqi/gh-aw-workflows"
HF_BASE = f"hf://datasets/{DATASET}"


## 1. Origen y carga de los datos

**Dataset:** <https://huggingface.co/datasets/luchosqi/gh-aw-workflows>

El dataset tiene tres tablas Parquet relacionadas 1:N:

| Tabla | Qué representa cada fila |
|---|---|
| `repositories` | Un repositorio de GitHub que usa GH-AW. |
| `workflow_files` | Un archivo `.md` de GH-AW (dentro de `.github/workflows/`) de un repositorio, con su body en Markdown ya separado del frontmatter. |
| `frontmatter_entries` | Una clave del frontmatter YAML de un archivo, aplanada a `(key_path, value, value_type)` — modelo entidad-atributo-valor, porque el frontmatter no tiene un esquema fijo. |

Se cargan directamente desde el Hub con `pd.read_parquet("hf://datasets/...")`.

In [2]:
repositories = pd.read_parquet(f"{HF_BASE}/repositories.parquet")
workflow_files = pd.read_parquet(f"{HF_BASE}/workflow_files.parquet")
frontmatter_entries = pd.read_parquet(f"{HF_BASE}/frontmatter_entries.parquet")

for name, df in [("repositories", repositories), ("workflow_files", workflow_files),
                  ("frontmatter_entries", frontmatter_entries)]:
    print(f"{name:22s} {df.shape[0]:>7,} filas  x  {df.shape[1]} columnas")


/home/luchosqi/Documentos/Universidad/semestres/Semestre 6/Ing Datos/Tarea1/Miner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repositories               370 filas  x  4 columnas
workflow_files           1,490 filas  x  5 columnas
frontmatter_entries     42,079 filas  x  5 columnas


In [3]:
repositories.head(3)

,repo_id,full_name,owner,name
0,0,azure/azure-sdk-for-java,azure,azure-sdk-for-java
1,1,activiti/activiti,activiti,activiti
2,2,apache/cloudstack,apache,cloudstack


In [4]:
workflow_files.head(3)

,file_id,repo_id,path,filename,body_markdown
0,0,0,.github/workflows/issue-triage.md,issue-triage.md,\n# Agentic Triage\n\n<!-- After editing this ...
1,1,0,.github/workflows/management-autopr-review.md,management-autopr-review.md,\n# Management AutoPR Review\n\nReview pull re...
2,2,1,.github/workflows/supply-chain-review.md,supply-chain-review.md,\n# Supply Chain Review\n\nYou are a supply ch...


In [5]:
frontmatter_entries.head(8)

,entry_id,file_id,key_path,value,value_type
0,0,0,description,Intelligent issue triage assistant that proces...,str
1,1,0,on.issues.types,"[""opened""]",json
2,2,0,on.workflow_dispatch.inputs.issue_number.descr...,Issue number to triage (used when dispatched f...,str
3,3,0,on.workflow_dispatch.inputs.issue_number.required,true,bool
4,4,0,on.workflow_dispatch.inputs.issue_number.type,string,str
5,5,0,on.roles,all,str
6,6,0,on.reaction,eyes,str
7,7,0,permissions.copilot-requests,write,str


## 2. Descripción de las tablas y sus relaciones

**Esquema:** `repositories (1) —< workflow_files (1) —< frontmatter_entries`

* `repositories.repo_id` — **PK**.
* `workflow_files.file_id` — **PK**; `workflow_files.repo_id` — **FK** → `repositories.repo_id`.
* `frontmatter_entries.entry_id` — **PK**; `frontmatter_entries.file_id` — **FK** → `workflow_files.file_id`.

In [6]:
def describe_table(name, df):
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_nulos": df.isna().sum(),
        "n_unicos": df.nunique(),
    })
    print(f"--- {name} ({len(df):,} filas) ---")
    display(info)

describe_table("repositories", repositories)
describe_table("workflow_files", workflow_files)
describe_table("frontmatter_entries", frontmatter_entries)


--- repositories (370 filas) ---


,dtype,n_nulos,n_unicos
repo_id,int64,0,370
full_name,str,0,370
owner,str,0,226
name,str,0,350


--- workflow_files (1,490 filas) ---


,dtype,n_nulos,n_unicos
file_id,int64,0,1490
repo_id,int64,0,370
path,str,0,1040
filename,str,0,1040
body_markdown,str,0,1306


--- frontmatter_entries (42,079 filas) ---


,dtype,n_nulos,n_unicos
entry_id,int64,0,42079
file_id,int64,0,1489
key_path,str,0,3184
value,str,0,8866
value_type,str,0,6


In [7]:
# Repositorios y archivos únicos, distinguidos del número de filas de cada tabla
n_repo_rows = len(repositories)
n_repo_ids_en_files = workflow_files["repo_id"].nunique()
n_file_rows = len(workflow_files)
n_file_ids_unicos = workflow_files["file_id"].nunique()

resumen_unicidad = pd.DataFrame({
    "concepto": [
        "Filas en repositories",
        "repo_id únicos en repositories",
        "repo_id únicos referenciados desde workflow_files",
        "Filas en workflow_files",
        "file_id únicos en workflow_files",
        "Filas en frontmatter_entries",
        "file_id únicos referenciados desde frontmatter_entries",
    ],
    "valor": [
        n_repo_rows,
        repositories["repo_id"].nunique(),
        n_repo_ids_en_files,
        n_file_rows,
        n_file_ids_unicos,
        len(frontmatter_entries),
        frontmatter_entries["file_id"].nunique(),
    ],
})
resumen_unicidad


,concepto,valor
0,Filas en repositories,370
1,repo_id únicos en repositories,370
2,repo_id únicos referenciados desde workflow_files,370
3,Filas en workflow_files,1490
4,file_id únicos en workflow_files,1490
5,Filas en frontmatter_entries,42079
6,file_id únicos referenciados desde frontmatter...,1489


**Lectura de la tabla anterior:** `repositories` tiene una fila por repositorio (370), así que
"filas" y "repositorios únicos" coinciden por construcción. En cambio `workflow_files` tiene **1 490
filas** que corresponden a solo **370 repo_id distintos** — es decir, en promedio cada repositorio
aporta más de un archivo GH-AW, y el número de filas *no* debe confundirse con el número de
repositorios. Del mismo modo, `frontmatter_entries` tiene 42 079 filas que provienen de únicamente
1 490 `file_id` distintos: varias filas (una por clave del YAML) describen un mismo archivo.

## 3. Revisión de calidad

Se revisan, para las tres tablas: valores ausentes por columna, filas duplicadas, claves
primarias repetidas o vacías, claves foráneas sin correspondencia, y consistencia de tipos /
formato en columnas derivadas (`full_name` vs. `owner`/`name`, `path` vs. `filename`, y que el
`value` de cada entrada del frontmatter sea parseable según su `value_type` declarado).

La ausencia de un campo del frontmatter (p. ej. que un archivo no declare `engine.id`) **no**
se trata como error: es información legítima — GH-AW aplica un motor de IA por defecto cuando la
clave no está presente.

In [8]:
# --- valores ausentes por columna ---
nulos = pd.concat({
    "repositories": repositories.isna().sum(),
    "workflow_files": workflow_files.isna().sum(),
    "frontmatter_entries": frontmatter_entries.isna().sum(),
}, names=["tabla", "columna"]).rename("n_nulos").reset_index()
nulos


,tabla,columna,n_nulos
0,repositories,repo_id,0
1,repositories,full_name,0
2,repositories,owner,0
3,repositories,name,0
4,workflow_files,file_id,0
5,workflow_files,repo_id,0
6,workflow_files,path,0
7,workflow_files,filename,0
8,workflow_files,body_markdown,0
9,frontmatter_entries,entry_id,0


In [9]:
# --- filas duplicadas y claves primarias repetidas / vacías ---
calidad_pk = pd.DataFrame([
    {"tabla": "repositories", "filas_duplicadas": repositories.duplicated().sum(),
     "pk": "repo_id", "pk_duplicadas": repositories["repo_id"].duplicated().sum(),
     "pk_vacias": repositories["repo_id"].isna().sum()},
    {"tabla": "workflow_files", "filas_duplicadas": workflow_files.duplicated().sum(),
     "pk": "file_id", "pk_duplicadas": workflow_files["file_id"].duplicated().sum(),
     "pk_vacias": workflow_files["file_id"].isna().sum()},
    {"tabla": "frontmatter_entries", "filas_duplicadas": frontmatter_entries.duplicated().sum(),
     "pk": "entry_id", "pk_duplicadas": frontmatter_entries["entry_id"].duplicated().sum(),
     "pk_vacias": frontmatter_entries["entry_id"].isna().sum()},
])
calidad_pk


,tabla,filas_duplicadas,pk,pk_duplicadas,pk_vacias
0,repositories,0,repo_id,0,0
1,workflow_files,0,file_id,0,0
2,frontmatter_entries,0,entry_id,0,0


In [10]:
# --- claves foráneas sin correspondencia ---
fk_huerfanas = pd.DataFrame([
    {"relación": "workflow_files.repo_id -> repositories.repo_id",
     "huérfanas": (~workflow_files["repo_id"].isin(repositories["repo_id"])).sum()},
    {"relación": "frontmatter_entries.file_id -> workflow_files.file_id",
     "huérfanas": (~frontmatter_entries["file_id"].isin(workflow_files["file_id"])).sum()},
])
fk_huerfanas


,relación,huérfanas
0,workflow_files.repo_id -> repositories.repo_id,0
1,frontmatter_entries.file_id -> workflow_files....,0


In [11]:
# --- consistencia de columnas derivadas ---
consistencia = pd.DataFrame([
    {"chequeo": "repositories.full_name == owner + '/' + name",
     "inconsistencias": (repositories["full_name"] != repositories["owner"] + "/" + repositories["name"]).sum()},
    {"chequeo": "workflow_files.path == '.github/workflows/' + filename",
     "inconsistencias": (workflow_files["path"] != ".github/workflows/" + workflow_files["filename"]).sum()},
    {"chequeo": "duplicado (repo_id, path) en workflow_files",
     "inconsistencias": workflow_files.duplicated(subset=["repo_id", "path"]).sum()},
])
consistencia


,chequeo,inconsistencias
0,repositories.full_name == owner + '/' + name,0
1,workflow_files.path == '.github/workflows/' + ...,0
2,"duplicado (repo_id, path) en workflow_files",0


In [12]:
# --- valor parseable según value_type declarado, por tipo ---
import json

def valor_parseable(value, value_type):
    try:
        if value_type == "int":
            return str(value).lstrip("-").isdigit()
        if value_type == "float":
            float(value); return True
        if value_type == "bool":
            return value in ("true", "false")
        if value_type == "null":
            return value == "None"
        if value_type == "json":
            json.loads(value); return True
        return True  # "str": cualquier texto es válido
    except Exception:
        return False

chequeo_valores = frontmatter_entries.assign(
    ok=frontmatter_entries.apply(lambda r: valor_parseable(r["value"], r["value_type"]), axis=1)
)
resumen_tipos = (
    chequeo_valores.groupby("value_type")["ok"]
    .agg(n_filas="count", n_invalidos=lambda s: (~s).sum())
    .reset_index()
)
resumen_tipos


,value_type,n_filas,n_invalidos
0,bool,6085,0
1,float,26,0
2,int,4035,0
3,json,8145,0
4,null,1793,0
5,str,21995,0


**Resultado de la revisión:** no se encontraron valores ausentes, filas duplicadas, claves
primarias repetidas o vacías, ni claves foráneas huérfanas en ninguna de las tres tablas. Las
columnas derivadas (`full_name`, `path`) son consistentes con sus partes en el 100% de las filas,
y el `value` de cada entrada del frontmatter es parseable según el `value_type` que se le asignó
al construir el dataset. Esto es coherente con el origen del dataset: se generó
programáticamente con `miner dataset` a partir de contenido descargado en el momento de la
corrida, sin edición manual posterior.

## 4. Tratamiento de los problemas encontrados

Las comprobaciones de la sección anterior no encontraron nulos, duplicados, PKs repetidas/vacías,
FKs huérfanas ni inconsistencias de formato — por lo tanto **no se excluye ni corrige ningún
registro**.

Sí se agrega una columna derivada que se usará repetidamente en el segundo notebook
(`body_word_count`, la longitud en palabras del body de cada archivo), para no repetir ese
cálculo y para dejar registrada su definición junto con las comprobaciones de calidad que la
preceden. Es una **transformación preparatoria**, no una corrección de errores.

In [13]:
workflow_files_proc = workflow_files.copy()
workflow_files_proc["body_word_count"] = (
    workflow_files_proc["body_markdown"].fillna("").str.split().str.len()
)

print("Filas modificadas (columna agregada a todas):", len(workflow_files_proc))
print("Filas excluidas: 0")
workflow_files_proc[["file_id", "repo_id", "filename", "body_word_count"]].head()


Filas modificadas (columna agregada a todas): 1490
Filas excluidas: 0


,file_id,repo_id,filename,body_word_count
0,0,0,issue-triage.md,3814
1,1,0,management-autopr-review.md,738
2,2,1,supply-chain-review.md,2593
3,3,2,daily-issue-triage.md,1020
4,4,2,weekly-repo-status.md,105


### Guardar las tablas preparadas para el segundo notebook

Se guardan las tres tablas en `eda/data/processed/`, conservando los datos originales
(`repositories` y `frontmatter_entries` se copian sin cambios; `workflow_files` incluye la
columna `body_word_count`).

In [14]:
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

repositories.to_parquet(PROCESSED_DIR / "repositories.parquet", index=False)
workflow_files_proc.to_parquet(PROCESSED_DIR / "workflow_files.parquet", index=False)
frontmatter_entries.to_parquet(PROCESSED_DIR / "frontmatter_entries.parquet", index=False)

sorted(p.name for p in PROCESSED_DIR.glob("*.parquet"))


['frontmatter_entries.parquet',
 'repositories.parquet',
 'workflow_files.parquet']